Data Processing:

Authorize Google Drive account:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Step 1: Examining Data

In [ ]:
# Load the data
PATH_TO_TWEETS = "Enter path to tweets tweets.txt"
PATH_TO_EMOJI = "Enter path to emoji.txt"
with open(PATH_TO_TWEETS) as f:
    tweets = f.readlines()

with open(PATH_TO_EMOJI, "r") as f:
    emojis = f.readlines()

# Basic structure check
print("Number of tweets:", len(tweets))
print("Number of emojis:", len(emojis))
print()
print("First 5 tweets:")
for t in tweets[:5]:
    print(repr(t))
print()
print("First 5 emojis:")
for e in emojis[:5]:
    print(repr(e))

Number of tweets: 225331
Number of emojis: 225331

First 5 tweets:
"RT @VibingOverHoes: Bet you'll get hungry  https://t.co/hOOU9xpmVq\n"
'Starbucks employee confuses boyfriend by saying is penis is tall \n'
'When your Starbucks store makes you an iced mocha instead of white mocha \n'
'Being told "girl your romper looks fierce!" At the subway today made me so happy, it is the little things that make all the difference \n'
'I got a Starbucks drink at school today, shit tasted soooo nasty I had to through that shit away \n'

First 5 emojis:
'heart_eyes\n'
'yum\n'
'sob\n'
'blush\n'
'sob\n'


Based on this response we can tell that the data needs to be cleaned more. The '\n' must be stripped from every emoji.

In [ ]:
#stripping '\n' from every emoji

tweets = [t.strip() for t in tweets]
emojis = [e.strip() for e in emojis]


RT @VibingOverHoes: Bet you'll get hungry  https://t.co/hOOU9xpmVq
heart_eyes


Now, we need to clean the tweet text to get rid of URLS, @mentions, retweet markers, and extra white space.

In [ ]:
def clean_tweet(tweet):
    tweet = re.sub(r'http\S+', '', tweet)        # remove URLs
    tweet = re.sub(r'@\w+', '', tweet)           # remove @mentions
    tweet = re.sub(r'^RT', '', tweet)            # remove RT at the start
    tweet = re.sub(r'^[\s:]+', '', tweet)        # remove leftover ': ' at the start
    tweet = re.sub(r'\s+', ' ', tweet)           # collapse extra whitespace
    tweet = tweet.strip()                        # trim edges
    return tweet

tweets_clean = [clean_tweet(t) for t in tweets]

for t in tweets_clean[:5]:
    print(t)

Bet you'll get hungry
Starbucks employee confuses boyfriend by saying is penis is tall
When your Starbucks store makes you an iced mocha instead of white mocha
Being told "girl your romper looks fierce!" At the subway today made me so happy, it is the little things that make all the difference
I got a Starbucks drink at school today, shit tasted soooo nasty I had to through that shit away


Sanity check to see if there are no empty tweets.

In [ ]:
empty = [(i, tweets_clean[i]) for i in range(len(tweets_clean)) if tweets_clean[i] == '']
print("Number of empty tweets:", len(empty))

Number of empty tweets: 0


We also need to make sure everything is lower case so the model treats them as the same word.

In [ ]:
tweets_clean = [t.lower() for t in tweets_clean]

# Check the first 5
for t in tweets_clean[:5]:
    print(t)

bet you'll get hungry
starbucks employee confuses boyfriend by saying is penis is tall
when your starbucks store makes you an iced mocha instead of white mocha
being told "girl your romper looks fierce!" at the subway today made me so happy, it is the little things that make all the difference
i got a starbucks drink at school today, shit tasted soooo nasty i had to through that shit away


Splitting the data into training and test sets so each model starts at the same baseline.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    tweets_clean, emojis,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 180264
Testing samples: 45067


Saving the cleaned split:

In [ ]:
import pickle

with open("/content/drive/My Drive/emoji_project/datasets/data_split.pkl", "wb") as f:
    pickle.dump((X_train, X_test, y_train, y_test), f)

print("Saved!")

Saved!


Verify inside the pickle file:

In [ ]:
import pickle

with open("/content/drive/My Drive/emoji_project/datasets/split_data/data_split.pkl", "rb") as f:
    X_train_check, X_test_check, y_train_check, y_test_check = pickle.load(f)

print("Training samples:", len(X_train_check))
print("Testing samples:", len(X_test_check))
print()
print("Sample tweet:", X_train_check[0])
print("Sample emoji:", y_train_check[0])

Training samples: 180264
Testing samples: 45067

Sample tweet: bet you'll get hungry
Sample emoji: heart_eyes
